# 🔐 NetworkSecurity — Phishing Detection ML Pipeline

**Full pipeline —

| Step | What happens |
|------|--------------|
| 1 | Mount Google Drive & set project root |
| 2 | Install dependencies |
| 3 | Set environment variables (MongoDB, MLflow) |
| 4 | Scaffold project folder structure |
| 5 | Write all `networksecurity` package files |
| 6 | Push raw CSV → MongoDB (one-time) |
| 7 | Run full training pipeline |
| 8 | Verify & display saved artifacts |
| 9 | Git commit instructions |

## ✅ Step 1 — Mount Google Drive & Set Project Root

In [2]:
import os, sys

# Everything stays in Colab's local storage — download before session ends
PROJECT_ROOT = '/content/NetworkSecurity'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)

# Make sure Python can find our package
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'✅ Working directory: {os.getcwd()}')

✅ Working directory: /content/NetworkSecurity


## ✅ Step 2 — Install Dependencies

In [1]:
%%capture install_log
!pip install python-dotenv pandas numpy pymongo certifi 'pymongo[srv]' \
             scikit-learn mlflow pyaml dagshub python-multipart scipy

In [3]:
import sklearn, mlflow, pymongo
print(f'✅ scikit-learn {sklearn.__version__} | mlflow {mlflow.__version__} | pymongo {pymongo.__version__}')

✅ scikit-learn 1.6.1 | mlflow 3.10.1 | pymongo 4.16.0


## ✅ Step 3 — Set Environment Variables
> Fill in your MongoDB URL. DagsHub MLflow is optional — set `USE_DAGSHUB = False` to log locally.

In [5]:
import os
from google.colab import userdata

# ── MongoDB (from Colab Secrets) ──────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────────────────────────
USE_DAGSHUB = True  # ✅ updated

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow — logs saved inside Colab
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

✅ Env vars set.
   MLFLOW_TRACKING_URI = https://dagshub.com/prithusarkar90/networksecurity.mlflow


## ✅ Step 4 — Scaffold Project Folder Structure

In [6]:
import os

dirs = [
    'networksecurity/exception',
    'networksecurity/logging',
    'networksecurity/constant',
    'networksecurity/entity',
    'networksecurity/components',
    'networksecurity/pipeline',
    'networksecurity/utils/main_utils',
    'networksecurity/utils/ml_utils/model',
    'networksecurity/utils/ml_utils/metric',
    'data_schema',
    'Network_Data',
    'final_model',
    'prediction_output',
    'logs',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# touch all __init__.py files
init_paths = [
    'networksecurity/__init__.py',
    'networksecurity/exception/__init__.py',
    'networksecurity/logging/__init__.py',
    'networksecurity/constant/__init__.py',
    'networksecurity/entity/__init__.py',
    'networksecurity/components/__init__.py',
    'networksecurity/pipeline/__init__.py',
    'networksecurity/utils/__init__.py',
    'networksecurity/utils/main_utils/__init__.py',
    'networksecurity/utils/ml_utils/__init__.py',
    'networksecurity/utils/ml_utils/model/__init__.py',
    'networksecurity/utils/ml_utils/metric/__init__.py',
]
for p in init_paths:
    if not os.path.exists(p):
        open(p, 'w').close()

print('✅ Project structure ready.')

✅ Project structure ready.


## ✅ Step 5 — Write All Package Files
> Each cell writes one module. Run all top-to-bottom once; skip on subsequent sessions if files already exist.

In [9]:
%%writefile networksecurity/exception/exception.py
import sys
# from networksecurity.logging import logger  # not needed here

class NetworkSecurityException(Exception):
    def __init__(self, error_message, error_details: sys):
        self.error_message = error_message
        _, _, exc_tb = error_details.exc_info()
        self.lineno    = exc_tb.tb_lineno
        self.file_name = exc_tb.tb_frame.f_code.co_filename

    def __str__(self):
        return (
            f'Error occurred in python script name [{self.file_name}] '
            f'line number [{self.lineno}] '
            f'error message [{str(self.error_message)}]'
        )

Overwriting networksecurity/exception/exception.py


In [10]:
%%writefile networksecurity/logging/logger.py
import logging
import os
from datetime import datetime

LOG_FILE = f"{datetime.now().strftime('%m_%d_%Y_%H_%M_%S')}.log"
logs_path = os.path.join(os.getcwd(), 'logs')
os.makedirs(logs_path, exist_ok=True)
LOG_FILE_PATH = os.path.join(logs_path, LOG_FILE)

logging.basicConfig(
    filename=LOG_FILE_PATH,
    format='[ %(asctime)s ] %(lineno)d %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO,
)

Writing networksecurity/logging/logger.py


In [11]:
%%writefile networksecurity/constant/training_pipeline.py
import os

# ── Pipeline ──────────────────────────────────────────────────────────────────
PIPELINE_NAME: str   = 'NetworkSecurity'
ARTIFACT_DIR: str    = 'Artifacts'
SAVED_MODEL_DIR: str = os.path.join('saved_models')

# ── File names ────────────────────────────────────────────────────────────────
FILE_NAME: str       = 'phisingData.csv'
TRAIN_FILE_NAME: str = 'train.csv'
TEST_FILE_NAME: str  = 'test.csv'
SCHEMA_FILE_PATH     = os.path.join('data_schema', 'schema.yaml')

# ── MongoDB ───────────────────────────────────────────────────────────────────
DATA_INGESTION_COLLECTION_NAME: str = 'NetworkData'
DATA_INGESTION_DATABASE_NAME: str   = 'KRISHAI'

# ── Data Ingestion ────────────────────────────────────────────────────────────
DATA_INGESTION_DIR_NAME: str          = 'data_ingestion'
DATA_INGESTION_FEATURE_STORE_DIR: str = 'feature_store'
DATA_INGESTION_INGESTED_DIR: str      = 'ingested'
DATA_INGESTION_TRAIN_TEST_SPLIT_RATION: float = 0.2

# ── Data Validation ───────────────────────────────────────────────────────────
DATA_VALIDATION_DIR_NAME: str               = 'data_validation'
DATA_VALIDATION_VALID_DIR: str              = 'validated'
DATA_VALIDATION_INVALID_DIR: str            = 'invalid'
DATA_VALIDATION_DRIFT_REPORT_DIR: str       = 'drift_report'
DATA_VALIDATION_DRIFT_REPORT_FILE_NAME: str = 'report.yaml'

# ── Data Transformation ───────────────────────────────────────────────────────
DATA_TRANSFORMATION_DIR_NAME: str               = 'data_transformation'
DATA_TRANSFORMATION_TRANSFORMED_DATA_DIR: str   = 'transformed'
DATA_TRANSFORMATION_TRANSFORMED_OBJECT_DIR: str = 'transformed_object'
PREPROCESSING_OBJECT_FILE_NAME: str             = 'preprocessing.pkl'
DATA_TRANSFORMATION_IMPUTER_PARAMS: dict = {
    'missing_values': float('nan'),
    'n_neighbors': 3,
    'weights': 'uniform',
}

# ── Target Column ─────────────────────────────────────────────────────────────
TARGET_COLUMN = 'Result'

# ── Model Trainer ─────────────────────────────────────────────────────────────
MODEL_TRAINER_DIR_NAME: str               = 'model_trainer'
MODEL_TRAINER_TRAINED_MODEL_DIR: str      = 'trained_model'
MODEL_FILE_NAME: str                      = 'model.pkl'
MODEL_TRAINER_EXPECTED_SCORE: float       = 0.6
MODEL_TRAINER_OVER_FIITING_UNDER_FITTING_THRESHOLD: float = 0.05

# ── Cloud stubs (not used in Colab — kept for import compatibility) ───────────
TRAINING_BUCKET_NAME = ''

Writing networksecurity/constant/training_pipeline.py


In [12]:
# Schema YAML — 30 feature columns + 1 target (Result)
schema_content = """having_IP_Address:\n  type: int\nURL_Length:\n  type: int\nShortining_Service:\n  type: int\nhaving_At_Symbol:\n  type: int\ndouble_slash_redirecting:\n  type: int\nPrefix_Suffix:\n  type: int\nhaving_Sub_Domain:\n  type: int\nSSLfinal_State:\n  type: int\nDomain_registeration_length:\n  type: int\nFavicon:\n  type: int\nport:\n  type: int\nHTTPS_token:\n  type: int\nRequest_URL:\n  type: int\nURL_of_Anchor:\n  type: int\nLinks_in_tags:\n  type: int\nSFH:\n  type: int\nSubmitting_to_email:\n  type: int\nAbnormal_URL:\n  type: int\nRedirect:\n  type: int\non_mouseover:\n  type: int\nRightClick:\n  type: int\npopUpWidnow:\n  type: int\nIframe:\n  type: int\nage_of_domain:\n  type: int\nDNSRecord:\n  type: int\nweb_traffic:\n  type: int\nPage_Rank:\n  type: int\nGoogle_Index:\n  type: int\nLinks_pointing_to_page:\n  type: int\nStatistical_report:\n  type: int\nResult:\n  type: int\n"""

with open('data_schema/schema.yaml', 'w') as f:
    f.write(schema_content)
print('✅ data_schema/schema.yaml written (31 columns).')

✅ data_schema/schema.yaml written (31 columns).


In [13]:
%%writefile networksecurity/entity/artifact_entity.py
from dataclasses import dataclass

@dataclass
class DataIngestionArtifact:
    trained_file_path: str
    test_file_path: str

@dataclass
class DataValidationArtifact:
    validation_status: bool
    valid_train_file_path: str
    valid_test_file_path: str
    invalid_train_file_path: str
    invalid_test_file_path: str
    drift_report_file_path: str

@dataclass
class DataTransformationArtifact:
    transformed_object_file_path: str
    transformed_train_file_path: str
    transformed_test_file_path: str

@dataclass
class ClassificationMetricArtifact:
    f1_score: float
    precision_score: float
    recall_score: float

@dataclass
class ModelTrainerArtifact:
    trained_model_file_path: str
    train_metric_artifact: ClassificationMetricArtifact
    test_metric_artifact: ClassificationMetricArtifact

Writing networksecurity/entity/artifact_entity.py


In [15]:
%%writefile networksecurity/entity/config_entity.py
from datetime import datetime
import os
from networksecurity.constant import training_pipeline

# print(training_pipeline.PIPELINE_NAME)  # debug print — not needed
# print(training_pipeline.ARTIFACT_DIR)   # debug print — not needed

class TrainingPipelineConfig:
    def __init__(self, timestamp=datetime.now()):
        timestamp = timestamp.strftime('%m_%d_%Y_%H_%M_%S')
        self.pipeline_name  = training_pipeline.PIPELINE_NAME
        self.artifact_name  = training_pipeline.ARTIFACT_DIR
        self.artifact_dir   = os.path.join(self.artifact_name, timestamp)
        self.model_dir      = os.path.join('final_model')
        self.timestamp: str = timestamp

class DataIngestionConfig:
    def __init__(self, training_pipeline_config: TrainingPipelineConfig):
        self.data_ingestion_dir: str = os.path.join(
            training_pipeline_config.artifact_dir, training_pipeline.DATA_INGESTION_DIR_NAME
        )
        self.feature_store_file_path: str = os.path.join(
            self.data_ingestion_dir,
            training_pipeline.DATA_INGESTION_FEATURE_STORE_DIR,
            training_pipeline.FILE_NAME,
        )
        self.training_file_path: str = os.path.join(
            self.data_ingestion_dir,
            training_pipeline.DATA_INGESTION_INGESTED_DIR,
            training_pipeline.TRAIN_FILE_NAME,
        )
        self.testing_file_path: str = os.path.join(
            self.data_ingestion_dir,
            training_pipeline.DATA_INGESTION_INGESTED_DIR,
            training_pipeline.TEST_FILE_NAME,
        )
        self.train_test_split_ratio: float = training_pipeline.DATA_INGESTION_TRAIN_TEST_SPLIT_RATION
        self.collection_name: str          = training_pipeline.DATA_INGESTION_COLLECTION_NAME
        self.database_name: str            = training_pipeline.DATA_INGESTION_DATABASE_NAME

class DataValidationConfig:
    def __init__(self, training_pipeline_config: TrainingPipelineConfig):
        self.data_validation_dir: str = os.path.join(
            training_pipeline_config.artifact_dir, training_pipeline.DATA_VALIDATION_DIR_NAME
        )
        self.valid_data_dir: str   = os.path.join(self.data_validation_dir, training_pipeline.DATA_VALIDATION_VALID_DIR)
        self.invalid_data_dir: str = os.path.join(self.data_validation_dir, training_pipeline.DATA_VALIDATION_INVALID_DIR)
        self.valid_train_file_path: str   = os.path.join(self.valid_data_dir,   training_pipeline.TRAIN_FILE_NAME)
        self.valid_test_file_path: str    = os.path.join(self.valid_data_dir,   training_pipeline.TEST_FILE_NAME)
        self.invalid_train_file_path: str = os.path.join(self.invalid_data_dir, training_pipeline.TRAIN_FILE_NAME)
        self.invalid_test_file_path: str  = os.path.join(self.invalid_data_dir, training_pipeline.TEST_FILE_NAME)
        self.drift_report_file_path: str  = os.path.join(
            self.data_validation_dir,
            training_pipeline.DATA_VALIDATION_DRIFT_REPORT_DIR,
            training_pipeline.DATA_VALIDATION_DRIFT_REPORT_FILE_NAME,
        )

class DataTransformationConfig:
    def __init__(self, training_pipeline_config: TrainingPipelineConfig):
        self.data_transformation_dir: str = os.path.join(
            training_pipeline_config.artifact_dir, training_pipeline.DATA_TRANSFORMATION_DIR_NAME
        )
        self.transformed_train_file_path: str = os.path.join(
            self.data_transformation_dir,
            training_pipeline.DATA_TRANSFORMATION_TRANSFORMED_DATA_DIR,
            training_pipeline.TRAIN_FILE_NAME.replace('csv', 'npy'),
        )
        self.transformed_test_file_path: str = os.path.join(
            self.data_transformation_dir,
            training_pipeline.DATA_TRANSFORMATION_TRANSFORMED_DATA_DIR,
            training_pipeline.TEST_FILE_NAME.replace('csv', 'npy'),
        )
        self.transformed_object_file_path: str = os.path.join(
            self.data_transformation_dir,
            training_pipeline.DATA_TRANSFORMATION_TRANSFORMED_OBJECT_DIR,
            training_pipeline.PREPROCESSING_OBJECT_FILE_NAME,
        )

class ModelTrainerConfig:
    def __init__(self, training_pipeline_config: TrainingPipelineConfig):
        self.model_trainer_dir: str = os.path.join(
            training_pipeline_config.artifact_dir, training_pipeline.MODEL_TRAINER_DIR_NAME
        )
        self.trained_model_file_path: str = os.path.join(
            self.model_trainer_dir,
            training_pipeline.MODEL_TRAINER_TRAINED_MODEL_DIR,
            training_pipeline.MODEL_FILE_NAME,
        )
        self.expected_accuracy: float = training_pipeline.MODEL_TRAINER_EXPECTED_SCORE
        self.overfitting_underfitting_threshold = training_pipeline.MODEL_TRAINER_OVER_FIITING_UNDER_FITTING_THRESHOLD

Overwriting networksecurity/entity/config_entity.py


In [35]:
%%writefile networksecurity/utils/main_utils/utils.py
import yaml
import os, sys
import numpy as np
import pickle
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from networksecurity.exception.exception import NetworkSecurityException
from networksecurity.logging.logger import logging

def convert_numpy_types_to_python(data):
    """Recursively converts NumPy types to standard Python types."""
    if isinstance(data, dict):
        return {k: convert_numpy_types_to_python(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [convert_numpy_types_to_python(elem) for elem in data]
    elif isinstance(data, (np.bool_, np.integer, np.floating)):
        return data.item() # Convert NumPy scalar to Python scalar
    elif isinstance(data, np.ndarray):
        return data.tolist() # Convert NumPy array to Python list
    return data

def read_yaml_file(file_path: str) -> dict:
    try:
        with open(file_path, 'rb') as yaml_file:
            return yaml.safe_load(yaml_file)
    except Exception as e:
        raise NetworkSecurityException(e, sys) from e

def write_yaml_file(file_path: str, content: object, replace: bool = False) -> None:
    try:
        if replace and os.path.exists(file_path):
            os.remove(file_path)
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        # Convert content to be YAML-safe before dumping
        safe_content = convert_numpy_types_to_python(content)
        with open(file_path, 'w') as file:
            yaml.dump(safe_content, file)
    except Exception as e:
        raise NetworkSecurityException(e, sys)

def save_numpy_array_data(file_path: str, array: np.array):
    try:
        dir_path = os.path.dirname(file_path)
        os.makedirs(dir_path, exist_ok=True)
        with open(file_path, 'wb') as file_obj:
            np.save(file_obj, array)
    except Exception as e:
        raise NetworkSecurityException(e, sys) from e

def save_object(file_path: str, obj: object) -> None:
    try:
        logging.info('Entered the save_object method of MainUtils class')
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, 'wb') as file_obj:
            pickle.dump(obj, file_obj)
        logging.info('Exited the save_object method of MainUtils class')
    except Exception as e:
        raise NetworkSecurityException(e, sys) from e

def load_object(file_path: str) -> object:
    try:
        if not os.path.exists(file_path):
            raise Exception(f'The file: {file_path} does not exist')
        with open(file_path, 'rb') as file_obj:
            return pickle.load(file_obj)
    except Exception as e:
        raise NetworkSecurityException(e, sys) from e

def load_numpy_array_data(file_path: str) -> np.array:
    try:
        with open(file_path, 'rb') as file_obj:
            return np.load(file_obj)
    except Exception as e:
        raise NetworkSecurityException(e, sys) from e

def evaluate_models(X_train, y_train, X_test, y_test, models, param):
    try:
        report = {}
        for i in range(len(list(models))):
            model = list(models.values())[i]
            para  = param[list(models.keys())[i]]
            gs = GridSearchCV(model, para, cv=3)
            gs.fit(X_train, y_train)
            model.set_params(**gs.best_params_)
            model.fit(X_train, y_train)
            y_train_pred = model.predict(X_train)
            y_test_pred  = model.predict(X_test)
            train_model_score = r2_score(y_train, y_train_pred)
            test_model_score  = r2_score(y_test, y_test_pred)
            report[list(models.keys())[i]] = test_model_score
        return report
    except Exception as e:
        raise NetworkSecurityException(e, sys)

Overwriting networksecurity/utils/main_utils/utils.py


In [17]:
%%writefile networksecurity/utils/ml_utils/model/estimator.py
from networksecurity.exception.exception import NetworkSecurityException
import sys

class NetworkModel:
    def __init__(self, preprocessor, model):
        try:
            self.preprocessor = preprocessor
            self.model = model
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def predict(self, x):
        try:
            x_transform = self.preprocessor.transform(x)
            y_hat = self.model.predict(x_transform)
            return y_hat
        except Exception as e:
            raise NetworkSecurityException(e, sys)

Writing networksecurity/utils/ml_utils/model/estimator.py


In [18]:
%%writefile networksecurity/utils/ml_utils/metric/classification_metric.py
from networksecurity.entity.artifact_entity import ClassificationMetricArtifact
from networksecurity.exception.exception import NetworkSecurityException
from sklearn.metrics import f1_score, precision_score, recall_score
import sys

def get_classification_score(y_true, y_pred) -> ClassificationMetricArtifact:
    try:
        model_f1_score        = f1_score(y_true, y_pred)
        model_recall_score    = recall_score(y_true, y_pred)
        model_precision_score = precision_score(y_true, y_pred)
        return ClassificationMetricArtifact(
            f1_score=model_f1_score,
            precision_score=model_precision_score,
            recall_score=model_recall_score,
        )
    except Exception as e:
        raise NetworkSecurityException(e, sys)

Writing networksecurity/utils/ml_utils/metric/classification_metric.py


In [19]:
%%writefile networksecurity/components/data_ingestion.py
import os, sys
import numpy as np
import pandas as pd
import pymongo
import certifi
from sklearn.model_selection import train_test_split
from networksecurity.exception.exception import NetworkSecurityException
from networksecurity.logging.logger import logging
from networksecurity.entity.config_entity import DataIngestionConfig
from networksecurity.entity.artifact_entity import DataIngestionArtifact

MONGO_DB_URL = os.getenv('MONGO_DB_URL')

class DataIngestion:
    def __init__(self, data_ingestion_config: DataIngestionConfig):
        try:
            self.data_ingestion_config = data_ingestion_config
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def export_collection_as_dataframe(self):
        try:
            ca = certifi.where()
            database_name   = self.data_ingestion_config.database_name
            collection_name = self.data_ingestion_config.collection_name
            mongo_client    = pymongo.MongoClient(MONGO_DB_URL, tlsCAFile=ca)
            collection      = mongo_client[database_name][collection_name]
            df = pd.DataFrame(list(collection.find()))
            if '_id' in df.columns:
                df = df.drop(columns=['_id'], axis=1)
            df.replace({'na': np.nan}, inplace=True)
            return df
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def export_data_into_feature_store(self, dataframe: pd.DataFrame):
        try:
            feature_store_file_path = self.data_ingestion_config.feature_store_file_path
            os.makedirs(os.path.dirname(feature_store_file_path), exist_ok=True)
            dataframe.to_csv(feature_store_file_path, index=False, header=True)
            return dataframe
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def split_data_as_train_test(self, dataframe: pd.DataFrame):
        try:
            train_set, test_set = train_test_split(
                dataframe, test_size=self.data_ingestion_config.train_test_split_ratio
            )
            logging.info('Performed train/test split on the dataframe')
            dir_path = os.path.dirname(self.data_ingestion_config.training_file_path)
            os.makedirs(dir_path, exist_ok=True)
            train_set.to_csv(self.data_ingestion_config.training_file_path, index=False, header=True)
            test_set.to_csv(self.data_ingestion_config.testing_file_path,   index=False, header=True)
            logging.info('Exported train and test file paths.')
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def initiate_data_ingestion(self):
        try:
            dataframe = self.export_collection_as_dataframe()
            dataframe = self.export_data_into_feature_store(dataframe)
            self.split_data_as_train_test(dataframe)
            return DataIngestionArtifact(
                trained_file_path=self.data_ingestion_config.training_file_path,
                test_file_path=self.data_ingestion_config.testing_file_path,
            )
        except Exception as e:
            raise NetworkSecurityException(e, sys)

Writing networksecurity/components/data_ingestion.py


In [20]:
%%writefile networksecurity/components/data_validation.py
import os, sys
import pandas as pd
from scipy.stats import ks_2samp
from networksecurity.entity.artifact_entity import DataIngestionArtifact, DataValidationArtifact
from networksecurity.entity.config_entity import DataValidationConfig
from networksecurity.exception.exception import NetworkSecurityException
from networksecurity.logging.logger import logging
from networksecurity.constant.training_pipeline import SCHEMA_FILE_PATH
from networksecurity.utils.main_utils.utils import read_yaml_file, write_yaml_file

class DataValidation:
    def __init__(self, data_ingestion_artifact: DataIngestionArtifact,
                 data_validation_config: DataValidationConfig):
        try:
            self.data_ingestion_artifact = data_ingestion_artifact
            self.data_validation_config  = data_validation_config
            self._schema_config = read_yaml_file(SCHEMA_FILE_PATH)
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    @staticmethod
    def read_data(file_path) -> pd.DataFrame:
        try:
            return pd.read_csv(file_path)
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def validate_number_of_columns(self, dataframe: pd.DataFrame) -> bool:
        try:
            number_of_columns = len(self._schema_config)
            logging.info(f'Required columns: {number_of_columns}, DataFrame columns: {len(dataframe.columns)}')
            return len(dataframe.columns) == number_of_columns
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def detect_dataset_drift(self, base_df, current_df, threshold=0.05) -> bool:
        try:
            status = True
            report = {}
            for column in base_df.columns:
                d1 = base_df[column]
                d2 = current_df[column]
                is_same_dist = ks_2samp(d1, d2)
                is_found = threshold > is_same_dist.pvalue
                if is_found:
                    status = False
                report[column] = {
                    'p_value': float(is_same_dist.pvalue),
                    'drift_status': is_found,
                }
            drift_report_file_path = self.data_validation_config.drift_report_file_path
            os.makedirs(os.path.dirname(drift_report_file_path), exist_ok=True)
            write_yaml_file(file_path=drift_report_file_path, content=report)
            return status
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def initiate_data_validation(self) -> DataValidationArtifact:
        try:
            train_file_path = self.data_ingestion_artifact.trained_file_path
            test_file_path  = self.data_ingestion_artifact.test_file_path
            train_dataframe = DataValidation.read_data(train_file_path)
            test_dataframe  = DataValidation.read_data(test_file_path)

            if not self.validate_number_of_columns(dataframe=train_dataframe):
                logging.warning('Train dataframe does not contain all columns.')
            if not self.validate_number_of_columns(dataframe=test_dataframe):
                logging.warning('Test dataframe does not contain all columns.')

            status = self.detect_dataset_drift(base_df=train_dataframe, current_df=test_dataframe)

            os.makedirs(os.path.dirname(self.data_validation_config.valid_train_file_path), exist_ok=True)
            train_dataframe.to_csv(self.data_validation_config.valid_train_file_path, index=False, header=True)
            test_dataframe.to_csv(self.data_validation_config.valid_test_file_path,   index=False, header=True)

            return DataValidationArtifact(
                validation_status=status,
                valid_train_file_path=self.data_ingestion_artifact.trained_file_path,
                valid_test_file_path=self.data_ingestion_artifact.test_file_path,
                invalid_train_file_path=None,
                invalid_test_file_path=None,
                drift_report_file_path=self.data_validation_config.drift_report_file_path,
            )
        except Exception as e:
            raise NetworkSecurityException(e, sys)

Writing networksecurity/components/data_validation.py


In [21]:
%%writefile networksecurity/components/data_transformation.py
import sys, os
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from networksecurity.constant.training_pipeline import TARGET_COLUMN, DATA_TRANSFORMATION_IMPUTER_PARAMS
from networksecurity.entity.artifact_entity import DataTransformationArtifact, DataValidationArtifact
from networksecurity.entity.config_entity import DataTransformationConfig
from networksecurity.exception.exception import NetworkSecurityException
from networksecurity.logging.logger import logging
from networksecurity.utils.main_utils.utils import save_numpy_array_data, save_object

class DataTransformation:
    def __init__(self, data_validation_artifact: DataValidationArtifact,
                 data_transformation_config: DataTransformationConfig):
        try:
            self.data_validation_artifact   = data_validation_artifact
            self.data_transformation_config = data_transformation_config
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    @staticmethod
    def read_data(file_path) -> pd.DataFrame:
        try:
            return pd.read_csv(file_path)
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def get_data_transformer_object(self) -> Pipeline:
        try:
            imputer: KNNImputer = KNNImputer(**DATA_TRANSFORMATION_IMPUTER_PARAMS)
            logging.info(f'Initialised KNNImputer with {DATA_TRANSFORMATION_IMPUTER_PARAMS}')
            return Pipeline([('imputer', imputer)])
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def initiate_data_transformation(self) -> DataTransformationArtifact:
        logging.info('Starting data transformation')
        try:
            train_df = DataTransformation.read_data(self.data_validation_artifact.valid_train_file_path)
            test_df  = DataTransformation.read_data(self.data_validation_artifact.valid_test_file_path)

            input_feature_train_df = train_df.drop(columns=[TARGET_COLUMN], axis=1)
            target_feature_train_df = train_df[TARGET_COLUMN].replace(-1, 0)

            input_feature_test_df  = test_df.drop(columns=[TARGET_COLUMN], axis=1)
            target_feature_test_df = test_df[TARGET_COLUMN].replace(-1, 0)

            preprocessor        = self.get_data_transformer_object()
            preprocessor_object = preprocessor.fit(input_feature_train_df)

            transformed_input_train = preprocessor_object.transform(input_feature_train_df)
            transformed_input_test  = preprocessor_object.transform(input_feature_test_df)

            train_arr = np.c_[transformed_input_train, np.array(target_feature_train_df)]
            test_arr  = np.c_[transformed_input_test,  np.array(target_feature_test_df)]

            save_numpy_array_data(self.data_transformation_config.transformed_train_file_path,  array=train_arr)
            save_numpy_array_data(self.data_transformation_config.transformed_test_file_path,   array=test_arr)
            save_object(self.data_transformation_config.transformed_object_file_path, preprocessor_object)

            # Also save to final_model for reuse
            save_object('final_model/preprocessor.pkl', preprocessor_object)

            return DataTransformationArtifact(
                transformed_object_file_path=self.data_transformation_config.transformed_object_file_path,
                transformed_train_file_path=self.data_transformation_config.transformed_train_file_path,
                transformed_test_file_path=self.data_transformation_config.transformed_test_file_path,
            )
        except Exception as e:
            raise NetworkSecurityException(e, sys)

Writing networksecurity/components/data_transformation.py


In [22]:
%%writefile networksecurity/components/model_trainer.py
import os, sys
from urllib.parse import urlparse
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from networksecurity.exception.exception import NetworkSecurityException
from networksecurity.logging.logger import logging
from networksecurity.entity.artifact_entity import DataTransformationArtifact, ModelTrainerArtifact
from networksecurity.entity.config_entity import ModelTrainerConfig
from networksecurity.utils.ml_utils.model.estimator import NetworkModel
from networksecurity.utils.main_utils.utils import save_object, load_object, load_numpy_array_data, evaluate_models
from networksecurity.utils.ml_utils.metric.classification_metric import get_classification_score

class ModelTrainer:
    def __init__(self, model_trainer_config: ModelTrainerConfig,
                 data_transformation_artifact: DataTransformationArtifact):
        try:
            self.model_trainer_config         = model_trainer_config
            self.data_transformation_artifact = data_transformation_artifact
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def track_mlflow(self, best_model, classificationmetric):
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        with mlflow.start_run():
            mlflow.log_metric('f1_score',      classificationmetric.f1_score)
            mlflow.log_metric('precision',     classificationmetric.precision_score)
            mlflow.log_metric('recall_score',  classificationmetric.recall_score)
            if tracking_url_type_store != 'file':
                mlflow.sklearn.log_model(best_model, 'model', registered_model_name=type(best_model).__name__)
            else:
                mlflow.sklearn.log_model(best_model, 'model')

    def train_model(self, X_train, y_train, x_test, y_test):
        models = {
            'Random Forest':      RandomForestClassifier(verbose=1),
            'Decision Tree':      DecisionTreeClassifier(),
            'Gradient Boosting':  GradientBoostingClassifier(verbose=1),
            'Logistic Regression': LogisticRegression(verbose=1),
            'AdaBoost':           AdaBoostClassifier(),
        }
        params = {
            'Decision Tree':      {'criterion': ['gini', 'entropy', 'log_loss']},
            'Random Forest':      {'n_estimators': [8, 16, 32, 128, 256]},
            'Gradient Boosting':  {'learning_rate': [.1, .01, .05, .001],
                                   'subsample':     [0.6, 0.7, 0.75, 0.85, 0.9],
                                   'n_estimators':  [8, 16, 32, 64, 128, 256]},
            'Logistic Regression': {},
            'AdaBoost':           {'learning_rate': [.1, .01, .001],
                                   'n_estimators':  [8, 16, 32, 64, 128, 256]},
        }

        model_report: dict = evaluate_models(
            X_train=X_train, y_train=y_train, X_test=x_test, y_test=y_test,
            models=models, param=params
        )

        best_model_score = max(sorted(model_report.values()))
        best_model_name  = list(model_report.keys())[list(model_report.values()).index(best_model_score)]
        best_model       = models[best_model_name]
        logging.info(f'Best model: {best_model_name} | Score: {best_model_score}')

        y_train_pred = best_model.predict(X_train)
        classification_train_metric = get_classification_score(y_true=y_train, y_pred=y_train_pred)
        self.track_mlflow(best_model, classification_train_metric)

        y_test_pred = best_model.predict(x_test)
        classification_test_metric = get_classification_score(y_true=y_test, y_pred=y_test_pred)
        self.track_mlflow(best_model, classification_test_metric)

        preprocessor = load_object(file_path=self.data_transformation_artifact.transformed_object_file_path)
        os.makedirs(os.path.dirname(self.model_trainer_config.trained_model_file_path), exist_ok=True)

        network_model = NetworkModel(preprocessor=preprocessor, model=best_model)
        save_object(self.model_trainer_config.trained_model_file_path, obj=network_model)
        save_object('final_model/model.pkl', best_model)

        return ModelTrainerArtifact(
            trained_model_file_path=self.model_trainer_config.trained_model_file_path,
            train_metric_artifact=classification_train_metric,
            test_metric_artifact=classification_test_metric,
        )

    def initiate_model_trainer(self) -> ModelTrainerArtifact:
        try:
            train_arr = load_numpy_array_data(self.data_transformation_artifact.transformed_train_file_path)
            test_arr  = load_numpy_array_data(self.data_transformation_artifact.transformed_test_file_path)
            x_train, y_train = train_arr[:, :-1], train_arr[:, -1]
            x_test,  y_test  = test_arr[:, :-1],  test_arr[:, -1]
            return self.train_model(x_train, y_train, x_test, y_test)
        except Exception as e:
            raise NetworkSecurityException(e, sys)

Writing networksecurity/components/model_trainer.py


In [23]:
%%writefile networksecurity/pipeline/training_pipeline.py
# ── Deployment-free version: S3 sync removed ─────────────────────────────────
import os, sys
from networksecurity.exception.exception import NetworkSecurityException
from networksecurity.logging.logger import logging
from networksecurity.components.data_ingestion import DataIngestion
from networksecurity.components.data_validation import DataValidation
from networksecurity.components.data_transformation import DataTransformation
from networksecurity.components.model_trainer import ModelTrainer
from networksecurity.entity.config_entity import (
    TrainingPipelineConfig, DataIngestionConfig, DataValidationConfig,
    DataTransformationConfig, ModelTrainerConfig,
)
from networksecurity.entity.artifact_entity import (
    DataIngestionArtifact, DataValidationArtifact,
    DataTransformationArtifact, ModelTrainerArtifact,
)

class TrainingPipeline:
    def __init__(self):
        self.training_pipeline_config = TrainingPipelineConfig()

    def start_data_ingestion(self) -> DataIngestionArtifact:
        try:
            config = DataIngestionConfig(training_pipeline_config=self.training_pipeline_config)
            logging.info('Starting Data Ingestion')
            artifact = DataIngestion(data_ingestion_config=config).initiate_data_ingestion()
            logging.info(f'Data Ingestion complete: {artifact}')
            return artifact
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def start_data_validation(self, data_ingestion_artifact: DataIngestionArtifact) -> DataValidationArtifact:
        try:
            config = DataValidationConfig(training_pipeline_config=self.training_pipeline_config)
            logging.info('Starting Data Validation')
            artifact = DataValidation(
                data_ingestion_artifact=data_ingestion_artifact,
                data_validation_config=config
            ).initiate_data_validation()
            logging.info(f'Data Validation complete: {artifact}')
            return artifact
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def start_data_transformation(self, data_validation_artifact: DataValidationArtifact) -> DataTransformationArtifact:
        try:
            config = DataTransformationConfig(training_pipeline_config=self.training_pipeline_config)
            logging.info('Starting Data Transformation')
            artifact = DataTransformation(
                data_validation_artifact=data_validation_artifact,
                data_transformation_config=config
            ).initiate_data_transformation()
            logging.info(f'Data Transformation complete: {artifact}')
            return artifact
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def start_model_trainer(self, data_transformation_artifact: DataTransformationArtifact) -> ModelTrainerArtifact:
        try:
            config = ModelTrainerConfig(training_pipeline_config=self.training_pipeline_config)
            logging.info('Starting Model Training')
            artifact = ModelTrainer(
                model_trainer_config=config,
                data_transformation_artifact=data_transformation_artifact
            ).initiate_model_trainer()
            logging.info(f'Model Training complete: {artifact}')
            return artifact
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def run_pipeline(self) -> ModelTrainerArtifact:
        try:
            ingestion_artifact      = self.start_data_ingestion()
            validation_artifact     = self.start_data_validation(ingestion_artifact)
            transformation_artifact = self.start_data_transformation(validation_artifact)
            model_artifact          = self.start_model_trainer(transformation_artifact)
            return model_artifact
        except Exception as e:
            raise NetworkSecurityException(e, sys)

Writing networksecurity/pipeline/training_pipeline.py


In [24]:
%%writefile push_data.py
# Run this ONCE to upload your CSV to MongoDB
import os, sys, json
import certifi
import pandas as pd
import numpy as np
import pymongo
from networksecurity.exception.exception import NetworkSecurityException
from networksecurity.logging.logger import logging

MONGO_DB_URL = os.getenv('MONGO_DB_URL')

class NetworkDataExtract:
    def __init__(self):
        pass

    def csv_to_json_convertor(self, file_path):
        try:
            data = pd.read_csv(file_path)
            data.reset_index(drop=True, inplace=True)
            records = list(json.loads(data.T.to_json()).values())
            return records
        except Exception as e:
            raise NetworkSecurityException(e, sys)

    def insert_data_mongodb(self, records, database, collection):
        try:
            ca = certifi.where()
            mongo_client = pymongo.MongoClient(MONGO_DB_URL, tlsCAFile=ca)
            db           = mongo_client[database]
            col          = db[collection]
            col.insert_many(records)
            return len(records)
        except Exception as e:
            raise NetworkSecurityException(e, sys)

if __name__ == '__main__':
    FILE_PATH  = 'Network_Data/phisingData.csv'
    DATABASE   = 'KRISHAI'
    COLLECTION = 'NetworkData'
    obj        = NetworkDataExtract()
    records    = obj.csv_to_json_convertor(file_path=FILE_PATH)
    count      = obj.insert_data_mongodb(records, DATABASE, COLLECTION)
    print(f'Inserted {count} records into MongoDB.')

Writing push_data.py


In [25]:
%%writefile setup.py
from setuptools import find_packages, setup
from typing import List

def get_requirements() -> List[str]:
    requirement_lst: List[str] = []
    try:
        with open('requirements.txt', 'r') as file:
            lines = file.readlines()
            for line in lines:
                requirement = line.strip()
                if requirement and requirement != '-e .':
                    requirement_lst.append(requirement)
    except FileNotFoundError:
        print('requirements.txt not found')
    return requirement_lst

setup(
    name='NetworkSecurity',
    version='0.0.1',
    author='Krish Naik',
    author_email='krishnaik06@gmail.com',
    packages=find_packages(),
    install_requires=get_requirements(),
)

Writing setup.py


In [26]:
%%writefile requirements.txt
python-dotenv
pandas
numpy
pymongo
certifi
pymongo[srv]
scikit-learn
mlflow
pyaml
dagshub
scipy
python-multipart
-e .

Writing requirements.txt


In [44]:
%%writefile .gitignore
# Secrets
.env

# Python
__pycache__/
*.py[cod]
*.egg-info/
dist/
build/

# Artifacts & models (large files — pull from Drive manually)
Artifacts/
final_model/
mlruns/
logs/
prediction_output/

# Jupyter checkpoints
.ipynb_checkpoints/

# Data
Network_Data/*.csv

Writing .gitignore


## ✅ Step 6 — (One-Time) Upload CSV to MongoDB

In [29]:
import shutil
import os

# Copy CSV from /content to the expected location
shutil.copy('/content/phisingData.csv', 'Network_Data/phisingData.csv')
print('✅ CSV copied to Network_Data/phisingData.csv')

✅ CSV copied to Network_Data/phisingData.csv


In [31]:
import os, sys, json, certifi
import pandas as pd
import pymongo

MONGO_DB_URL = os.environ['MONGO_DB_URL']

FILE_PATH  = 'Network_Data/phisingData.csv'
DATABASE   = 'KRISHAI'
COLLECTION = 'NetworkData'

if os.path.exists(FILE_PATH):
    # Convert CSV to records
    data = pd.read_csv(FILE_PATH)
    data.reset_index(drop=True, inplace=True)
    records = list(json.loads(data.T.to_json()).values())

    # Connect with certifi SSL fix
    ca = certifi.where()
    client = pymongo.MongoClient(MONGO_DB_URL, tlsCAFile=ca, tls=True)
    col = client[DATABASE][COLLECTION]
    col.insert_many(records)
    print(f'✅ Inserted {len(records)} records into MongoDB.')
else:
    print(f'⚠️ {FILE_PATH} not found.')

✅ Inserted 11055 records into MongoDB.


## ✅ Step 7 — Run the Full Training Pipeline

In [37]:
import importlib
import sys

# Reload modules so edits above are picked up without restarting runtime
mods_to_reload = [
    'networksecurity.exception.exception',
    'networksecurity.logging.logger',
    'networksecurity.constant.training_pipeline',
    'networksecurity.entity.artifact_entity',
    'networksecurity.entity.config_entity',
    'networksecurity.utils.main_utils.utils',
    'networksecurity.utils.ml_utils.model.estimator',
    'networksecurity.utils.ml_utils.metric.classification_metric',
    'networksecurity.components.data_ingestion',
    'networksecurity.components.data_validation',
    'networksecurity.components.data_transformation',
    'networksecurity.components.model_trainer',
    'networksecurity.pipeline.training_pipeline',
]
for mod in mods_to_reload:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from networksecurity.pipeline.training_pipeline import TrainingPipeline

pipeline = TrainingPipeline()
model_trainer_artifact = pipeline.run_pipeline()

print('\n✅ Pipeline complete!')
print(f'   Train F1:     {model_trainer_artifact.train_metric_artifact.f1_score:.4f}')
print(f'   Test  F1:     {model_trainer_artifact.test_metric_artifact.f1_score:.4f}')
print(f'   Test  Prec:   {model_trainer_artifact.test_metric_artifact.precision_score:.4f}')
print(f'   Test  Recall: {model_trainer_artifact.test_metric_artifact.recall_score:.4f}')

[Parallel(n_jobs=1)]: Done   8 out of   8 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   8 out of   8 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   8 out of   8 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   8 out of   8 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   8 out of   8 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   8 out of   8 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  16 out of  16 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done  16 out of  16 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  16 out of  16 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done  16 out of  16 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  16 out of  16 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done  16 out of  16 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  32 out of  32 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done  32 out of  32 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]

Streaming output truncated to the last 5000 lines.
        30           0.3771          -0.0735            0.02s
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           1.2353           0.1364            0.34s
         2           1.1229           0.1108            0.32s
         3           1.0303           0.0964            0.31s
         4           0.9509           0.0673            0.30s
         5           0.8895           0.0946            0.29s
         6           0.8294           0.0381            0.27s
         7           0.7757           0.0246            0.26s
         8           0.7347           0.0508            0.26s
         9           0.7021           0.0572            0.26s
        10           0.6701           0.0272            0.25s
        20           0.4839           0.0607            0.15s
        30           0.3926           0.0196            0.02s
      Iter       Train Loss      OOB Improve   Remaining Time 
         1       

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  32 out of  32 | elapsed:    0.0s finished
2026/03/17 06:11:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 06:11:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RandomForestClassifier' already exists. Creating a new

🏃 View run rebellious-fly-616 at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0/runs/9cfc898ae15b4d96b5202b6f7588052c
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0


[Parallel(n_jobs=1)]: Done  32 out of  32 | elapsed:    0.0s finished
2026/03/17 06:11:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 06:11:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RandomForestClassifier' already exists. Creating a new version of this model...
2026/03/17 06:11:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForestClassifier, version 4
Created version '4' of model 'RandomForestClassifier'.


🏃 View run serious-hen-779 at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0/runs/55593ffc3f8240d7af2564eb9d470a69
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0

✅ Pipeline complete!
   Train F1:     0.9908
   Test  F1:     0.9722
   Test  Prec:   0.9618
   Test  Recall: 0.9829


## ✅ Step 8 — Verify & Display Saved Artifacts

In [38]:
import os

important_files = [
    'final_model/model.pkl',
    'final_model/preprocessor.pkl',
]

print('=== Final Model Files ===')
for f in important_files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) if exists else 0
    status = f'✅ {size/1024:.1f} KB' if exists else '❌ NOT FOUND'
    print(f'  {f:<40} {status}')

print('\n=== Artifacts Directory ===')
for root, dirs, files in os.walk('Artifacts'):
    level = root.replace('Artifacts', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in files:
        fpath = os.path.join(root, file)
        size  = os.path.getsize(fpath)
        print(f'{subindent}{file}  ({size/1024:.1f} KB)')

=== Final Model Files ===
  final_model/model.pkl                    ✅ 3191.5 KB
  final_model/preprocessor.pkl             ✅ 2333.1 KB

=== Artifacts Directory ===
Artifacts/
  03_17_2026_06_06_21/
    data_transformation/
      transformed_object/
        preprocessing.pkl  (2333.1 KB)
      transformed/
        train.npy  (2142.0 KB)
        test.npy  (535.6 KB)
    data_validation/
      drift_report/
        report.yaml  (1.9 KB)
      validated/
        test.csv  (154.2 KB)
        train.csv  (616.9 KB)
    model_trainer/
      trained_model/
        model.pkl  (5524.6 KB)
    data_ingestion/
      feature_store/
        phisingData.csv  (770.7 KB)
      ingested/
        test.csv  (154.2 KB)
        train.csv  (616.9 KB)
  03_17_2026_05_59_07/
    data_transformation/
      transformed_object/
        preprocessing.pkl  (2333.1 KB)
      transformed/
        train.npy  (2142.0 KB)
        test.npy  (535.6 KB)
    data_validation/
      drift_report/
        report.yaml  (2.2 KB)

In [41]:
import yaml, glob
import os
from datetime import datetime

drift_files = glob.glob('Artifacts/**/report.yaml', recursive=True)

# Sort files by the timestamp in their path to get the latest one
def get_timestamp_from_path(path):
    parts = path.split(os.sep)
    for part in parts:
        if len(part) == 17 and part.count('_') == 4 and part.replace('_', '').isdigit():
            try:
                return datetime.strptime(part, '%m_%d_%Y_%H_%M_%S')
            except ValueError:
                continue
    return datetime.min # Return a very old date if no timestamp found

if drift_files:
    # Sort by timestamp in descending order
    drift_files.sort(key=get_timestamp_from_path, reverse=True)
    latest_drift_report_path = drift_files[0]

    with open(latest_drift_report_path) as f:
        report = yaml.safe_load(f)
    drifted = [col for col, v in report.items() if v.get('drift_status')]
    print(f'Drift report: {latest_drift_report_path}')
    print(f'Total columns checked : {len(report)}')
    print(f'Columns with drift    : {len(drifted)}')
    if drifted:
        print(f'Drifted columns       : {drifted}')
    else:
        print('✅ No significant drift detected.')
else:
    print('No drift report found yet — run the pipeline first.')

Drift report: Artifacts/03_17_2026_06_06_21/data_validation/drift_report/report.yaml
Total columns checked : 31
Columns with drift    : 0
✅ No significant drift detected.


In [45]:
import shutil

# Zip the entire project folder
shutil.make_archive('/content/NetworkSecurity_project', 'zip', '/content/NetworkSecurity')
print('✅ Download ready!')

# Download it
from google.colab import files
files.download('/content/NetworkSecurity_project.zip')

✅ Download ready!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
import os

must_have = [
    'final_model/model.pkl',
    'final_model/preprocessor.pkl',
    'data_schema/schema.yaml',
    'requirements.txt',
    'setup.py',
    '.gitignore',
    'push_data.py',
]

for f in must_have:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'{status} {f}')

✅ final_model/model.pkl
✅ final_model/preprocessor.pkl
✅ data_schema/schema.yaml
✅ requirements.txt
✅ setup.py
✅ .gitignore
✅ push_data.py
